# Massage data

pay careful attention to the options here: most are hacks for earlier versions of the pipeline

if the pipeline completed in one pass, with seq method calling information,
this can be probably be skipped.

In [ ]:
MULTIPLE_STATSFILES = True  # True if multiple runs need to be merged
SEQTYPE_CALL_FIXUP  = True  # old ver: True to merge recovered seq types with stats
FASTP_PRICE_FIXUP   = True  # old ver: fix missing fastp/price frameshift mistake
REVERSED_SC_FIXUP   = True  # old ver: fix reversed scRNAseq read order miscalled as PE

%store -r SCRIPTS DATA INTERMEDIATE STATS_FILES SEQTYPE_CALLS

In [ ]:
out_table = INTERMEDIATE/"stats_merged.csv"

if MULTIPLE_STATSFILES:
    ALL_STATS_FILES = " ".join(f.as_posix() for f in DATA.glob(STATS_FILES))
    !echo "$SCRIPTS/01_merge_stats.sh > $INTERMEDIATE/stats_merged.csv"
    !$SCRIPTS/01_merge_stats.sh $ALL_STATS_FILES > $out_table
else: # untested
    !echo "cp $STATS_FILES $INTERMEDIATE/stats_merged.csv"
    !cp $STATS_FILES $out_table

!head -n3 $INTERMEDIATE/stats_merged.csv


In [ ]:
if SEQTYPE_CALL_FIXUP:
    in_table = out_table
    out_table = INTERMEDIATE/"stats_merged_seqtype.csv"
    !echo "python3 $SCRIPTS/02_add_seqmethod_calls.py $in_table $SEQTYPE_CALLS > $out_table"
    !python3 $SCRIPTS/02_add_seqmethod_calls.py $in_table $SEQTYPE_CALLS > $out_table

if FASTP_PRICE_FIXUP:
    in_table = out_table
    out_table = INTERMEDIATE/"stats_merged_seqtype_fixup.csv"
    !echo "$SCRIPTS/02.5_fixup_missing_fastp_price.sh < $in_table > $out_table"
    !$SCRIPTS/02a_fixup_missing_fastp_price.sh < $in_table > $out_table

if REVERSED_SC_FIXUP:
    in_table = out_table
    out_table = INTERMEDIATE/"stats_merged_seqtype_revsc_fixup.csv"
    !echo "$SCRIPTS/02.6_fixup_old_reverse_sc_calls.sh < $in_table > $out_table"
    !$SCRIPTS/02b_fixup_old_reverse_sc_calls.sh < $in_table > $out_table

In [ ]:
!head -n3 $out_table
!cp $out_table $INTERMEDIATE/"stats_merged_complete.csv"

STATS_TABLE = INTERMEDIATE/"stats_merged_complete.csv"
%store STATS_TABLE